# Рекуррентные нейронные сети (RNN) с нуля с помощью numpy
В этой тетрадке мы напишем свою простую рекуррентную нейронную сеть (RNN) руками с помощью библиотеки numpy и обучим её генерировать новые (или реальные) имена людей на английском языке.

*С опорой на статью https://quantdare.com/implementing-a-rnn-with-numpy/*

Для этого мы построим RNN типа 'многие-ко-многим' ('many-to-many'): на вход подаётся последовательность символов — имя из n букв, и в качестве аутпута модель рекурсивно пытается для каждого символа (буквы) предсказать следующий.

Мы создадим класс RNNModel, который будет содержать следующие методы:

* **прямой проход** (forward propagation);

* **вычисление функции потерь** (loss calculation);

* **обратное распространение ошибки** (backward propagation);

* **усечение градиентов** (clipping) для борьбы с проблемой взрывающихся градиентов.

При желании можно добавить и другие методы: например, случайную инициализацию параметров или метод оптимизации.

## Подготовка данных
Мы будем использовать датасет 'person_names.txt', который содержит около 19.000 имён на английском. Каждое имя будем рассматривать как последовательность символов. Для обучения создаём пары (вход, правильный ответ),
где вход — символ в позиции `t`, а ответ — символ в позиции `t+1`.  

In [ ]:
import numpy as np

# Загружаем файл с именами
with open("person_names.txt", "r") as f:
    names = f.read().splitlines()

# Приводим к нижнему регистру
names = [name.lower() for name in names if name.strip() != ""]

# Создание пар (входная строка, целевая строка со сдвигом на 1 символ)
data = []
for name in names:
    input_seq = name[:-1]
    target_seq = name[1:]
    data.append((input_seq, target_seq))

# Формируем алфавит (все символы + спецсимвол конца строки)
alphabet = sorted(list(set("".join(names)))) + ["\n"]
print("Пример имени:", names[0])
print("Алфавит:", alphabet)

## Функции активации и softmax

Реализуем базовые функции, которые будут использоваться в нашей RNN:

- **tanh**, **sigmoid**, **ReLU** — функции активации;
- их производные — для обратного распространения ошибки;
- **softmax** — для преобразования выходных значений в вероятностное распределение;
- **negative log likelihood** — для функции потерь.

---

**Задание:** Реализуйте функцию `softmax(x)` самостоятельно.

In [ ]:
# ВАШ КОД ЗДЕСЬ
def softmax(x):
    ### Реализуйте softmax
    pass

In [ ]:
# Решение
def softmax(x):
    z = x - np.max(x)
    sm = np.exp(z) / np.sum(np.exp(z), axis=0)
    return sm

def deriv_tanh(tanh_value):
    return 1 - tanh_value * tanh_value

def sigmoid(x):
    return 1. / (1. + np.exp(-x))

def deriv_sigmoid(sigmoid_value):
    return sigmoid_value * (1. - sigmoid_value)

def relu(x):
    return np.maximum(x, 0)

def deriv_relu(relu_value):
    return 1. * (relu_value > 0)

def neg_log(x):
    return -np.log(x)

## Реализация RNN-модели

Теперь создадим класс `NNModel`, который будет содержать:

- инициализацию параметров (веса и смещения);
- функции активации;
- преобразование символа в one-hot вектор;
- прямой и обратный проход (forward и backward propagation);
- функцию обучения;
- функцию генерации имён.

---

**Задание:** Попробуйте самостоятельно реализовать шаг прямого прохода в методе `process_batch`.

In [ ]:
class NNModel:
    def __init__(self, alphabet, hidden_size, activation="tanh", learning_rate=0.1):
        self.hidden_size = hidden_size
        self.learning_rate = learning_rate
        self.HIDDEN_ACTIVATION = activation
        self.alphabet = alphabet
        self.alphabet_size = len(alphabet)
        self.char_to_ix = {ch: i for i, ch in enumerate(self.alphabet)}
        self.ix_to_char = {i: ch for i, ch in enumerate(self.alphabet)}
        self.W_ih = np.random.randn(self.hidden_size, self.alphabet_size) * 0.01
        self.W_hh = np.random.randn(self.hidden_size, self.hidden_size) * 0.01
        self.W_ho = np.random.randn(self.alphabet_size, self.hidden_size) * 0.01

    def hidden_activation(self, x):
        if self.HIDDEN_ACTIVATION == "tanh":
            return np.tanh(x)
        elif self.HIDDEN_ACTIVATION == "sigmoid":
            return sigmoid(x)
        elif self.HIDDEN_ACTIVATION == "relu":
            return relu(x)

    def deriv_hidden_activation(self, activation):
        if self.HIDDEN_ACTIVATION == "tanh":
            return deriv_tanh(activation)
        elif self.HIDDEN_ACTIVATION == "sigmoid":
            return deriv_sigmoid(activation)
        elif self.HIDDEN_ACTIVATION == "relu":
            return deriv_relu(activation)

    def alphabet_position_to_onehot_encode(self, x):
        onehot_encoded = np.zeros((self.alphabet_size, 1))
        onehot_encoded[x] = 1
        return onehot_encoded

    def process_batch(self, xs, ys, hprev):
        input_, hidden_out_, out_out_ = {}, {}, {}
        hidden_out_[-1] = np.copy(hprev)
        losses = []
        for t in range(len(xs)):
            input_[t] = self.alphabet_position_to_onehot_encode(xs[t])
            # ВАШ КОД ЗДЕСЬ: реализуйте прямой проход
            pass

In [ ]:
# Решение с прямым и обратным проходом
class NNModel:
    def __init__(self, alphabet, hidden_size, activation="tanh", learning_rate=0.1):
        self.hidden_size = hidden_size
        self.learning_rate = learning_rate
        self.HIDDEN_ACTIVATION = activation
        self.alphabet = alphabet
        self.alphabet_size = len(alphabet)
        self.char_to_ix = {ch: i for i, ch in enumerate(self.alphabet)}
        self.ix_to_char = {i: ch for i, ch in enumerate(self.alphabet)}
        self.W_ih = np.random.randn(self.hidden_size, self.alphabet_size) * 0.01
        self.W_hh = np.random.randn(self.hidden_size, self.hidden_size) * 0.01
        self.W_ho = np.random.randn(self.alphabet_size, self.hidden_size) * 0.01

    def hidden_activation(self, x):
        if self.HIDDEN_ACTIVATION == "tanh":
            return np.tanh(x)
        elif self.HIDDEN_ACTIVATION == "sigmoid":
            return sigmoid(x)
        elif self.HIDDEN_ACTIVATION == "relu":
            return relu(x)

    def deriv_hidden_activation(self, activation):
        if self.HIDDEN_ACTIVATION == "tanh":
            return deriv_tanh(activation)
        elif self.HIDDEN_ACTIVATION == "sigmoid":
            return deriv_sigmoid(activation)
        elif self.HIDDEN_ACTIVATION == "relu":
            return deriv_relu(activation)

    def alphabet_position_to_onehot_encode(self, x):
        onehot_encoded = np.zeros((self.alphabet_size, 1))
        onehot_encoded[x] = 1
        return onehot_encoded

    def process_batch(self, xs, ys, hprev):
        input_, hidden_out_, out_out_ = {}, {}, {}
        hidden_out_[-1] = np.copy(hprev)
        losses = []

        # Прямой проход
        for t in range(len(xs)):
            input_[t] = self.alphabet_position_to_onehot_encode(xs[t])
            h_in = self.W_ih @ input_[t] + self.W_hh @ hidden_out_[t - 1]
            hidden_out_[t] = self.hidden_activation(h_in)
            out = self.W_ho @ hidden_out_[t]
            out_out_[t] = softmax(out)
            losses.append(neg_log(out_out_[t][ys[t], 0]))

        # Обратный проход
        dW_ih = np.zeros_like(self.W_ih)
        dW_hh = np.zeros_like(self.W_hh)
        dW_ho = np.zeros_like(self.W_ho)
        dh_next = np.zeros((self.hidden_size, 1))

        for t in reversed(range(len(xs))):
            dy = np.copy(out_out_[t])
            dy[ys[t]] -= 1
            dW_ho += dy @ hidden_out_[t].T
            dh = self.W_ho.T @ dy + dh_next
            dh_raw = self.deriv_hidden_activation(hidden_out_[t]) * dh
            dW_ih += dh_raw @ input_[t].T
            dW_hh += dh_raw @ hidden_out_[t - 1].T
            dh_next = self.W_hh.T @ dh_raw

        return losses, dW_ih, dW_hh, dW_ho, hidden_out_[len(xs) - 1]

## Обучение модели и генерация имён

In [ ]:
def train(self, training_data, epochs=20):
    h_prev = np.zeros((self.hidden_size, 1))
    for epoch in range(epochs):
        total_loss = 0
        for input_seq, target_seq in training_data:
            xs = [self.char_to_ix[ch] for ch in input_seq]
            ys = [self.char_to_ix[ch] for ch in target_seq]
            losses, dW_ih, dW_hh, dW_ho, h_prev = self.process_batch(xs, ys, h_prev)
            self.W_ih -= self.learning_rate * dW_ih
            self.W_hh -= self.learning_rate * dW_hh
            self.W_ho -= self.learning_rate * dW_ho
            total_loss += sum(losses)
        if epoch % 5 == 0:
            print(f"Epoch {epoch+1}, Loss: {total_loss:.3f}")

def sample(self, seed_char, length=10):
    idx = self.char_to_ix[seed_char]
    x = self.alphabet_position_to_onehot_encode(idx)
    h = np.zeros((self.hidden_size, 1))
    result = [seed_char]
    for _ in range(length):
        h = self.hidden_activation(self.W_ih @ x + self.W_hh @ h)
        y = softmax(self.W_ho @ h).ravel()
        idx = np.random.choice(range(self.alphabet_size), p=y)
        x = self.alphabet_position_to_onehot_encode(idx)
        result.append(self.ix_to_char[idx])
    return ''.join(result)